<a href="https://colab.research.google.com/github/Ligohtml/Big_imaging/blob/main/Forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter

import matplotlib.pyplot as plt
from IPython.core.debugger import Tracer

# Definition of parameters

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [ ]:
df_test = pd.read_csv('/content/drive/MyDrive/ForestDataset/test.csv')
df_train = pd.read_csv('/content/drive/MyDrive/ForestDataset/train.csv')
df_val = pd.read_csv('/content/drive/MyDrive/ForestDataset/val.csv')
df_train.head()

,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,...,Soil_Type32,Soil_Type33,Soil_Type34,Soil_Type35,Soil_Type36,Soil_Type37,Soil_Type38,Soil_Type39,Soil_Type40,Cover_Type
0,-0.623282,1.601743,0.177250,-0.491248,-0.197208,-0.947926,-1.201046,-0.217778,0.891365,-0.092870,...,-0.218671,-0.206085,-0.038173,-0.082413,-0.025726,-0.047474,-0.224908,-0.213134,-0.176939,6
1,-0.127668,-1.386934,-0.414210,-0.938722,-0.670775,0.819591,-0.350268,-0.217778,0.281259,4.557553,...,-0.218671,-0.206085,-0.038173,-0.082413,-0.025726,-0.047474,-0.224908,-0.213134,-0.176939,1
2,1.589021,-0.151493,0.058958,-0.443645,-0.736095,1.929761,0.991343,0.615511,-0.481374,2.510094,...,-0.218671,-0.206085,-0.038173,-0.082413,-0.025726,-0.047474,-0.224908,-0.213134,-0.176939,7
3,0.298511,-0.369512,0.177250,-0.353198,-0.556466,0.501861,1.154954,0.045366,-0.895374,-0.861121,...,-0.218671,-0.206085,-0.038173,-0.082413,-0.025726,-0.047474,-0.224908,-0.213134,-0.176939,5
4,0.602583,-0.905475,0.177250,-0.224668,-0.376837,2.233152,0.533231,-0.831781,-0.830005,0.320804,...,-0.218671,-0.206085,-0.038173,-0.082413,-0.025726,-0.047474,-0.224908,-0.213134,-0.176939,1


In [ ]:
# How many classes are there?
classes = len(df_test['Cover_Type'].unique())
print(f"The unique values in Cover_Type column are: {classes} ")

df_train.shape

The unique values in Cover_Type column are: 7 


(12096, 55)

In [ ]:
device = 'cuda'
learning_rate = 0.0001
batch_size = 100
experiment_name = 'prova1'

# Data loading

In [ ]:
class Dataset(torch.utils.data.Dataset):

    def __init__(self, csv):
        # read the csv file
        self.df = pd.read_csv(csv)
        # self.df = self.df.dropna(axis=0)
        # save cols
        self.output_cols = ['Cover_Type']
        self.input_cols = list(set(self.df.columns)-set(self.output_cols))




    def __len__(self):
        # here i will return the number of samples in the dataset
        return len(self.df)


    def __getitem__(self, idx):
        # here i will load the file in position idx
        cur_sample = self.df.iloc[idx]
        if cur_sample.isna().any():
            Tracer()()
        # split in input / ground-truth
        cur_sample_x = cur_sample[self.input_cols]
        cur_sample_y = cur_sample[self.output_cols]
        # convert to torch format
        cur_sample_x = torch.tensor(cur_sample_x.tolist())
        cur_sample_y = torch.tensor(cur_sample_y.tolist()) - 1 # Subtract 1 to make labels 0-indexed
        # return values
        return cur_sample_x, cur_sample_y

In [ ]:
# try to use the dataset
ds = Dataset('/content/drive/MyDrive/ForestDataset/test.csv')
# get first item
xx,yy = ds.__getitem__(0)
# print shapes
print(xx.shape)
print(yy.shape)

torch.Size([54])
torch.Size([1])


In [ ]:
# create train and validation datasets
train_ds = Dataset('/content/drive/MyDrive/ForestDataset/train.csv')
val_ds =  Dataset('/content/drive/MyDrive/ForestDataset/val.csv')

In [ ]:
# create train dataloader
train_dl = torch.utils.data.DataLoader(
    train_ds,
    batch_size = batch_size,
    drop_last = True,
    shuffle = True,
    num_workers = 8
)
# create validation dataloader
val_dl = torch.utils.data.DataLoader(
    val_ds,
    batch_size = batch_size,
    drop_last = False,
    shuffle = False,
    num_workers = 8
)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


# Network definition

In [ ]:
# define network

class Net(nn.Module):

    def __init__(self):
        # initialize super class
        super(Net, self).__init__()
        self.layer1 = nn.Linear(54,64)
        self.layer2 = nn.ReLU()
        self.layer3 = nn.Linear(64,32)
        self.layer4 = nn.ReLU()
        self.layer5 = nn.Linear(32,16)
        self.layer6 = nn.ReLU()
        self.layer7 = nn.Linear(16, classes)


    def forward(self, x):
        # apply layers in cascade
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.layer5(x)
        x = self.layer6(x)
        x = self.layer7(x)
        # return output
        return x


In [ ]:
# let's test the network
net = Net()

# define random batch of 10 elements
inp = torch.rand(10, 54)

# forward
out = net(inp)

# let's print the shape
print(' Input shape is', inp.shape)
print('Output shape is', out.shape)

 Input shape is torch.Size([10, 54])
Output shape is torch.Size([10, 7])


In [ ]:
# let's move the network in GPU
net.to(device)

# define random batch of 10 elements
inp = torch.rand(10, 54)

# move the batch in GPU
inp = inp.to(device)

# get the output
out = net(inp)

# let's print the shape
print(' Input shape is', inp.shape)
print('Output shape is', out.shape)

 Input shape is torch.Size([10, 54])
Output shape is torch.Size([10, 7])


# Define validation routine

In [ ]:
# create validation routine
def validate(net, dl):
    # get final score
    score = 0
    # set network in eval mode
    net.eval()
    # at the end of epoch, validate model
    for inp, gt in dl:
        # move batch to gpu
        inp = inp.to(device)
        gt = gt.to(device)
        gt.squeeze()
        # get output
        with torch.no_grad():
            out = net(inp)
        # compare with gt
        cur_score = F.l1_loss(out, gt)
        # append
        score += cur_score
    # at the end, average over batches
    score /= len(dl)
    # set network in training mode
    net.train()
    # return score
    return score

# TO IMPLEMENT
def validation(network, data_loader):
  macro_acc = torchmetrics.Accuracy(task = 'multiclass', num_classes = 7, average = 'macro').to(device)
  micro_acc = torchmetrics.Accuracy(task = 'multiclass', num_classes = 7, average = 'micro').to(device)
  micro_acc.reset()
  macro_acc.reset()
  network.eval()
  #get logits torch.tensor
  # maximum = torch.max (logits, axis)
  for input, gt in data_loader:
    input = input.to(device)
    gt = gt.to(device)

    with torch.no_grad():
      output = network(input)
    # deve fare la predizione e solo dopo dopo valutare
    # output = network(input)
    macro_acc.update(output, gt.squeeze() )
    micro_acc.update(output, gt.squeeze() )

  micro = micro_acc.compute()
  macro = macro_acc.compute()

  print(f'Micro accuracy is {micro:0.2f} while macro accuracy is {macro:0.2f}')

  return macro



# Train

In [ ]:
import shutil
# %load_ext tensorboard
%reload_ext tensorboard
%tensorboard --logdir={experiment_name}

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


tensor([5, 4, 5, 2, 3, 5, 4, 4, 5, 2, 5, 1, 4, 4, 1, 5, 3, 2, 4, 5, 5, 2, 4, 4,
        1, 1, 6, 1, 1, 0, 4, 2, 1, 6, 0, 4, 0, 1, 0, 2, 6, 0, 5, 3, 1, 6, 5, 0,
        1, 0, 6, 4, 0, 6, 6, 6, 5, 3, 0, 0, 6, 4, 5, 3, 2, 1, 1, 3, 5, 3, 2, 2,
        4, 2, 5, 2, 3, 1, 1, 1, 1, 5, 4, 6, 4, 5, 3, 5, 2, 1, 2, 0, 5, 6, 5, 4,
        4, 3, 4, 4], dtype=torch.int32)
tensor([2, 1, 4, 1, 0, 1, 5, 3, 1, 6, 3, 5, 2, 0, 0, 0, 2, 5, 4, 4, 4, 5, 4, 0,
        4, 4, 0, 1, 2, 0, 0, 0, 3, 3, 5, 6, 4, 5, 4, 1, 6, 6, 6, 2, 0, 5, 2, 2,
        4, 0, 1, 1, 3, 1, 5, 4, 0, 0, 5, 5, 3, 2, 4, 2, 6, 1, 5, 6, 1, 2, 3, 5,
        3, 3, 5, 6, 4, 3, 6, 1, 3, 4, 0, 1, 5, 2, 1, 3, 3, 0, 6, 4, 4, 3, 3, 3,
        4, 0, 1, 3], dtype=torch.int32)
tensor([0, 0, 6, 0, 5, 1, 5, 1, 3, 4, 4, 5, 5, 0, 4, 2, 0, 0, 3, 4, 6, 0, 0, 5,
        4, 2, 6, 2, 0, 6, 1, 0, 3, 6, 4, 3, 2, 0, 5, 1, 1, 6, 3, 3, 4, 0, 6, 0,
        0, 4, 0, 2, 2, 1, 3, 2, 0, 1, 2, 2, 3, 4, 1, 5, 1, 3, 3, 2, 0, 1, 3, 3,
        6, 5, 3, 3, 5, 5, 6, 3, 4, 5, 0,

KeyboardInterrupt: 

In [ ]:
# define optimizer
optimizer = torch.optim.Adam(params=net.parameters(), lr=learning_rate)

# define summary writer
writer = SummaryWriter(experiment_name)

# initialize iteration number
n_iter = 0

# define best validation value
best_val = None

# for each epoch
for cur_epoch in range(50):
    # plot current epoch
    writer.add_scalar("epoch", cur_epoch, n_iter)
    # for each batch
    for inp, gt in train_dl:
        # move batch to gpu
        inp = inp.to(device)
        gt = gt.to(device)
        # reset gradients
        optimizer.zero_grad()
        # get output
        out = net(inp)
        # compute loss
        loss = nn.CrossEntropyLoss()
        output= loss(out,gt.squeeze().long())
        # compute backward
        output.backward()
        # update weights
        optimizer.step()
        # print
        # print(loss.item())
        # plot
        writer.add_scalar("loss", output.item(), n_iter)
        n_iter += 1

    # at the end, validate model
    cur_val = validate(net, val_dl)
    # plot validation
    print(f"\repoch: {cur_epoch}", end="", flush=True)
    writer.add_scalar("val", cur_val.item(), n_iter)
    # check if it is the best model so far
    if best_val is None or cur_val > best_val:
        # define new best val
        best_val = cur_val
        # save current model as best
        torch.save({
            'net': net.state_dict(),
            'opt': optimizer.state_dict(),
            'epoch': cur_epoch
        }, experiment_name + '_best.pth')
    # save last model
        torch.save({
            'net': net.state_dict(),
            'opt': optimizer.state_dict(),
            'epoch': cur_epoch
        }, experiment_name + '_last.pth')

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/tmp/ipykernel_4640/1495305.py:17: UserWarning: Using a target size (torch.Size([100, 1])) that is different to the input size (torch.Size([100, 7])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  cur_score = F.l1_loss(out, gt)


epoch: 0

/tmp/ipykernel_4640/1495305.py:17: UserWarning: Using a target size (torch.Size([12, 1])) that is different to the input size (torch.Size([12, 7])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  cur_score = F.l1_loss(out, gt)


epoch: 4

KeyboardInterrupt: 

# Test

In [ ]:
# create test dataset
test_ds =  Dataset('/content/drive/MyDrive/Big_imaging/Uber/test.csv')

# create dataloader
test_dl = torch.utils.data.DataLoader(
    test_ds,
    batch_size = batch_size,
    drop_last = False,
    shuffle = False,
    num_workers = 8
)

In [ ]:
# load best network
state = torch.load(experiment_name + '_best.pth')
net.load_state_dict(state['net'])

In [ ]:
test_value = validate(net, test_dl).item()

In [ ]:
print(f'The model scored a MAE of {test_value:0.04f} over the testset.')

ADD THE CONFUJSION MATRIx